# 單調讀取一致性（Monotonic Reads）— 基本款 + 進階款

這本 notebook 讓你自己動手驗證 **單調讀取一致性**。可以逐格執行、隨意改參數。

> **單調讀取（MR）的定義**：一個 client 一旦讀到某個值，之後的讀取就不能再看到更舊的狀態。
> 白話：讀到的版本只能往前，不能倒退。

我們做兩款：

| | 情境 | 機制 | 用到的旋鈕 |
|---|---|---|---|
| **基本款** | 網路分區 | rollback（回滾） | readConcern |
| **進階款** | 正常運作、無分區 | 讀到落後的從節點 | readPreference（第三個旋鈕）+ 因果 session |

**前置需求**：`./scripts/up.sh` 已經把 5 台叢集開起來（mongo1 是 primary）。


## 0. 環境準備

這本 notebook 從 **host** 直接連到每個節點的 localhost port。

> 注意：從 host 不能用 `?replicaSet=rs0`——driver 會把節點換成容器主機名（`mongo1:27017`…），host 解析不到。
> 所以我們一律用 `directConnection=true` 連 `localhost:<port>`，這樣就能指定讀哪一台。


In [ ]:
# 如果還沒裝 pymongo，取消下面這行的註解執行一次：
# %pip install pymongo

import subprocess, time
from pathlib import Path
from pymongo import MongoClient, ReadPreference
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern
from pymongo.errors import PyMongoError, ExecutionTimeout

# 每個節點在 host 上的 port（來自 docker-compose.yml）
NODES = {'mongo1':27017,'mongo2':27018,'mongo3':27019,'mongo4':27020,'mongo5':27021}
DB = 'consistency'

# 專案根目錄下的 scripts/（分區工具在這裡）。請確認這個路徑指到你的 repo。
REPO = Path.home()/'Projects'/'consistency-models'
SCRIPTS = REPO/'scripts'

def direct(node, socket_ms=5000):
    '''直連單一節點（不做 replica-set 探索）。'''
    return MongoClient(f'mongodb://localhost:{NODES[node]}/?directConnection=true',
                       serverSelectionTimeoutMS=3000, socketTimeoutMS=socket_ms)

def sh(script, *args):
    '''呼叫 scripts/ 底下的 shell 腳本（分區、修復）。'''
    subprocess.run([str(SCRIPTS/script), *args], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 冒煙測試：看得到 primary 就代表連得上
print('primary =', direct('mongo1').admin.command('hello').get('primary'))


## 1. 一些共用小工具

- `who_is_primary()`：現在誰是 primary
- `wait_primary(node, timeout)`：**等**某台變成 primary（事件屏障，不用 sleep 硬猜）
- `write(node, key, v, wc)` / `read(node, key, rc, session)`：對指定節點寫 / 讀


In [ ]:
def who_is_primary():
    try:
        return direct('mongo1').admin.command('hello').get('primary')
    except PyMongoError:
        return None

def wait_primary(node, timeout=40):
    '''輪詢直到 node 自認是可寫的 primary，或逾時。'''
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if direct(node, socket_ms=2000).admin.command('hello').get('isWritablePrimary'):
                return True
        except PyMongoError:
            pass
        time.sleep(2)
    return False

def write(node, key, v, wc=1, session=None, socket_ms=3000):
    c = direct(node, socket_ms=socket_ms)
    coll = c[DB].get_collection('mr', write_concern=WriteConcern(w=wc, wtimeout=3000))
    coll.update_one({'k':key}, {'$set':{'v':v}}, upsert=True, session=session)

def read(node, key, rc='local', session=None, socket_ms=5000, max_ms=5000):
    c = direct(node, socket_ms=socket_ms)
    coll = c[DB].get_collection('mr', read_concern=ReadConcern(rc))
    doc = coll.find_one({'k':key}, session=session, max_time_ms=max_ms)
    return doc['v'] if doc else None

print('工具就緒')


---
# 基本款：用網路分區 + rollback 讓單調讀取倒退

**想法**：MongoDB 只有一個 primary、寫入排成一條隊列，所以要讓讀取「倒退」，唯一辦法是讓一筆已經被讀到的值**被回滾掉**。

步驟：
1. 寫基準值 `k=0`（durable）
2. 把 `mongo1+mongo2`（少數邊，mongo1 是舊 primary）跟多數邊切開
3. 對 mongo1 用 `w:1` 寫 `k=1` → 回報成功，但這筆注定被回滾
4. **第一次讀** `k`：
   - `readConcern=local` → 讀到 1（那個注定消失的值）
   - `readConcern=majority` → 讀到 0（只給已被多數確認的）
5. 多數邊選出新 primary（mongo3），修復分區 → mongo1 那筆被丟掉
6. **第二次讀** `k` → 讀到 0
7. 若「第一次 1 → 第二次 0」= 讀取倒退 = **違反**

預期（對照 MongoDB 文件那張表：MR 只在 `readConcern=majority` 成立）：

| readConcern | 預測 |
|---|---|
| local | VIOLATED |
| majority | SAFE |

（writeConcern 對 MR 沒有影響——這也是要示範的重點。）


In [ ]:
def basic_monotonic_reads(read_concern='local'):
    key = f'mr-basic-{read_concern}-{int(time.time())}'
    # 1. 基準值 durable
    write('mongo1', key, 0, wc='majority')
    print(f'baseline {key}=0 (durable)')
    # 2. 分區
    sh('partition-split.sh', 'mongo1', 'mongo2')
    print('partitioned: [mongo1,mongo2] | [mongo3,mongo4,mongo5]')
    read1 = None
    try:
        # 3. 注定被回滾的 w:1 寫入
        doomed_acked = False
        try:
            write('mongo1', key, 1, wc=1, socket_ms=3000)
            doomed_acked = True
            print('doomed write k=1 w:1 ACKED on mongo1')
        except PyMongoError as e:
            print('doomed write refused:', str(e)[:60])
        # 4. 第一次讀（同一個因果 session）
        c = direct('mongo1', socket_ms=6000)
        with c.start_session(causal_consistency=True) as s:
            try:
                # session 只能配它自己開出來的 client，所以這裡直接用 c 讀
                coll = c[DB].get_collection('mr', read_concern=ReadConcern(read_concern))
                doc = coll.find_one({'k':key}, session=s, max_time_ms=4000)
                read1 = doc['v'] if doc else None
                print(f'READ 1 (readConcern={read_concern}) on mongo1 -> k={read1}')
            except PyMongoError as e:
                print('READ 1 blocked/unavailable:', type(e).__name__)
        # 5. 等新 primary + 修復
        print('waiting for mongo3 to be elected...')
        wait_primary('mongo3', 40)
    finally:
        sh('heal-split.sh')
        time.sleep(12)
    # 6. 第二次讀（分區已修復，讀存活下來的歷史）
    read2 = None
    for node in ('mongo1','mongo3'):
        try:
            read2 = read(node, key, rc=read_concern, max_ms=4000)
            break
        except PyMongoError:
            continue
    print(f'READ 2 (readConcern={read_concern}) -> k={read2}')
    # 7. 判定
    if read1 is not None and read2 is not None and read1 > read2:
        verdict = 'VIOLATED（讀取倒退：先讀到 %s，後讀到 %s）' % (read1, read2)
    elif read1 == read2:
        verdict = 'SAFE（兩次讀取一致，沒有倒退）'
    else:
        verdict = 'INCONCLUSIVE（檢查故障窗口時機）'
    print('=> verdict:', verdict)
    return verdict


### 跑 `readConcern=local` → 預期 VIOLATED


In [ ]:
basic_monotonic_reads('local')


### 跑 `readConcern=majority` → 預期 SAFE

只差一個字（`local` → `majority`），結果就從違反變安全。


In [ ]:
basic_monotonic_reads('majority')


---
# 進階款：不用分區，讀到落後的從節點就讓單調讀取倒退

基本款要製造分區。**進階款更貼近真實**：在**正常運作**下，只要你的 `readPreference` 把讀取送到一個落後的從節點，時間就倒退了——這才是實務上最常見的單調讀取違反。

我們刻意讓 `mongo5` 變成**延遲 10 秒的從節點**（`secondaryDelaySecs`），然後：

- **NAIVE（沒有因果 session）**：先讀一個最新的從節點（看到大版本），再讀落後的 mongo5（看到小版本）→ 倒退 → **違反**
- **CAUSAL（因果 session）**：同樣的讀取，但 session 帶著上一次讀到的 clusterTime → 讀 mongo5 時它會**阻塞等到自己追上**才回答 → 不會倒退 → **安全**（但那次讀會卡好幾秒）

這款示範兩件事：readPreference 是控制「讀哪一台」的旋鈕，而因果 session 是把單調性補回來的機制（代價是延遲）。

> 小知識：延遲從節點會被 MongoDB 從 `hello().hosts` 藏起來，所以一般 `readPreference=secondary` 的自動路由**不會**選到它。我們用 `directConnection` 直接釘上 mongo5 來穩定重現這個情況。


In [ ]:
def set_delay(node, secs):
    '''把某台設成延遲從節點（secondaryDelaySecs）。secs=0 還原。'''
    js = ('const c=rs.conf();'
          f'const i=c.members.findIndex(m=>m.host==="{node}:27017");'
          'c.version++;'
          f'if({secs}>0){{c.members[i].priority=0;c.members[i].secondaryDelaySecs={secs};}}'
          f'else{{c.members[i].secondaryDelaySecs=0;}}'
          'rs.reconfig(c);')
    subprocess.run(['docker','exec','mongo1','mongosh','--quiet','--eval',js],
                   capture_output=True, text=True)
    print(f'{node}: secondaryDelaySecs={secs}')

def fresh_secondary(primary_host):
    '''挑一個不是 primary、也不是延遲節點 mongo5 的從節點來當「最新」的一台。'''
    p = (primary_host or '').split(':')[0]
    for n in ('mongo2','mongo3','mongo4'):
        if n != p:
            return n
    return 'mongo2'


In [ ]:
def advanced_monotonic_reads(causal=False, n_writes=6):
    key = f'mr-adv-{"causal" if causal else "naive"}-{int(time.time())}'
    prim = who_is_primary(); pnode = (prim or 'mongo1:27017').split(':')[0]
    fresh = fresh_secondary(prim)
    # 先在「還沒延遲」時寫基準值 0，並等 mongo5 真的套用，
    # 這樣稍後 mongo5 落後時會停在一個『舊數字』(0)，而不是完全沒有這個 key。
    set_delay('mongo5', 0); time.sleep(2)
    write(pnode, key, 0, wc='majority', socket_ms=8000)
    for _ in range(10):
        try:
            if read('mongo5', key, rc='local', max_ms=3000) == 0: break
        except PyMongoError: pass
        time.sleep(1)
    set_delay('mongo5', 10); time.sleep(3)   # 現在起 mongo5 落後 10 秒
    try:
        print(f'primary={prim}  fresh secondary={fresh}  lagging=mongo5(+10s)')
        # 對 primary 連續寫新版本 1..n（mongo5 會延遲，最新從節點會拿到）
        for v in range(1, n_writes+1):
            write(pnode, key, v, wc='majority', socket_ms=8000)
            time.sleep(0.3)
        print(f'wrote {key}=1..{n_writes} on {pnode}')
        time.sleep(2)
        # naive 用弱讀 (local, 無 session)；causal 用強讀 (majority + 因果 session)
        rc = 'majority' if causal else 'local'
        cfresh = direct(fresh, socket_ms=8000)
        clag   = direct('mongo5', socket_ms=15000)
        sf = cfresh.start_session(causal_consistency=True) if causal else None
        sl = clag.start_session(causal_consistency=True) if causal else None
        # 讀 1：最新從節點
        vf = cfresh[DB].get_collection('mr', read_concern=ReadConcern(rc)).find_one(
                 {'k':key}, session=sf, max_time_ms=8000)
        v_fresh = vf['v'] if vf else None
        print(f'READ fresh ({fresh}) -> k={v_fresh}')
        # 因果模式：把 fresh 讀到的 clusterTime 帶到 mongo5 的 session（跨 client 允許）
        if causal and sf is not None:
            sl.advance_cluster_time(sf.cluster_time)
            sl.advance_operation_time(sf.operation_time)
        # 讀 2：落後的 mongo5
        t0 = time.time()
        try:
            vl = clag[DB].get_collection('mr', read_concern=ReadConcern(rc)).find_one(
                     {'k':key}, session=sl, max_time_ms=13000)
            v_lag = vl['v'] if vl else None
        except ExecutionTimeout:
            v_lag = 'TIMEOUT'
        dt = time.time() - t0
        print(f'READ lag (mongo5) -> k={v_lag}   （這次讀花了 {dt:.1f}s）')
        # 判定：讀到比 fresh 更舊（或變成不存在）= 倒退 = 違反
        if v_fresh is not None and (v_lag is None or (isinstance(v_lag,int) and v_lag < v_fresh)):
            verdict = f'VIOLATED（先讀到 {v_fresh}，再讀到更舊的 {v_lag}）'
        else:
            verdict = f'SAFE（第二次讀 = {v_lag}，沒有倒退）'
        print('=> verdict:', verdict)
        return verdict
    finally:
        set_delay('mongo5', 0)       # 還原
        time.sleep(2)


### NAIVE（沒有因果 session）→ 預期 VIOLATED

讀最新從節點看到大版本，讀 mongo5 看到舊版本，時間倒退。


In [ ]:
advanced_monotonic_reads(causal=False)


### CAUSAL（因果 session）→ 預期 SAFE（但那次讀會卡住等 mongo5 追上）

注意輸出裡「這次讀花了 N 秒」——因果一致性不是讓 mongo5 變新，是讓你的讀取**阻塞等它追上**。這就是保證的代價。


In [ ]:
advanced_monotonic_reads(causal=True)


---
## 收尾與延伸

跑完後把 mongo5 的延遲還原（上面的 `finally` 已自動處理），叢集回到乾淨狀態。

**你可以自己改的地方**：
- 基本款把 `readConcern` 換成 `local` / `majority` 對照
- 進階款把 `n_writes`、mongo5 的延遲秒數調大調小，看違反是否更容易出現
- 進階款把讀取的 `read_concern` 從 `majority` 改成 `local`，觀察差異
- 把 `direct('mongo5')` 換成別的從節點，看哪些會落後

**對應報告**：這本 notebook 對應單調讀取那一節。基本款覆蓋「網路分區」情境、進階款覆蓋「正常運作」情境，兩者都是作業要求⑤點名的 scenario。進階款還額外示範了 readPreference（第三個旋鈕）與因果 session 的取捨，正好補強要求②的「探索 consistency configurations」。
